# Industry classification — study version

This version executes **the same code** as the clean `industry_classification.ipynb` notebook. Only comments and explanations differ. The submission is a self-contained notebook, matching the course format, rather than a separate Python package. Comments focus on what is learned, from which data, and why; explaining basic Python syntax is not the main purpose of the oral exam.

Run every cell from top to bottom using the `.venv` Python kernel. The two course datasets must be in `data/`. Embeddings are supplied: we do not train a language model or need a GPU. Synthetic-data checks verify program behavior, not performance on the course dataset. Historical outputs in `archive/` are not newly measured results for this version.

**The project in one sentence.** Two classifiers predict an industry from the meaning of a business description; a separate model learns how much to trust that prediction using agreement, ambiguity and financial compatibility.

An embedding matrix `X` has one row per company and one column per embedding coordinate. A label vector `y` has one industry per company. A probability matrix has one row per company and one column per industry. Company order must match across these structures.

In [ ]:
# ast.literal_eval converts embeddings stored as text into lists of numbers without executing arbitrary code.
import ast
# A monotonic clock measures the elapsed duration of this run.
from time import perf_counter
# NumPy handles embedding matrices, probabilities, Boolean masks and vectorized calculations.
import numpy as np
# pandas handles tables, joins by company ID, missing values and summaries.
import pandas as pd
# Matplotlib builds the figures, titles and axes; it does not train the models.
import matplotlib.pyplot as plt
# Seaborn provides readable heatmaps, boxplots and industry colors.
import seaborn as sns
# display renders formatted tables inside the notebook.
from IPython.display import display
# PCA learns a linear projection for compressing and visualizing the embeddings.
from sklearn.decomposition import PCA
# Logistic regression will classify industries and, separately, estimate prediction correctness.
from sklearn.linear_model import LogisticRegression
# These metrics measure overall correctness, F1, performance by class and classification errors.
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
# Group-based splitting keeps all records of the same company in the same subset.
from sklearn.model_selection import GroupShuffleSplit
# k-NN uses the labels of nearby fitting companies in the PCA space.
from sklearn.neighbors import KNeighborsClassifier
# LabelEncoder maps industry names to integer class codes.
from sklearn.preprocessing import LabelEncoder
# Thread control limits the resources used by numerical libraries.
from threadpoolctl import threadpool_limits

# Two threads limit CPU use without changing the learning objective.
threadpool_limits(limits=2)
# Start the timer before loading data so the final duration includes data preparation and plotting.
started = perf_counter()


## 1. Data and independent roles

Each description is already represented by a numerical vector: descriptions with similar meaning tend to have similar representations. Individual coordinates are not identifiable words or financial ratios. Company ID links tables and identifies repeated companies; it is not a numerical feature from which the classifier should learn industries.

Join finances by ID using a **left join**, retaining companies with missing information. Multiple financial records with the same ID are summarized by their median. This prevents duplicate join matches from multiplying companies and reduces sensitivity to extreme observations. A negative margin can represent a genuine loss, so we keep it. An infinite value is unusable and becomes `NaN`. We do not replace unknown margins with zero, which has a different economic meaning.

| Subset | Approximate fraction | Learns or decides |
|---|---:|---|
| Fitting | 81% | PCA, industry classifiers and financial reference profiles |
| Tuning | 4.5% | The classifiers’ voting weight |
| Calibration | 4.5% | Probability that a fixed industry prediction is correct |
| Validation | 10% | Metrics of the completed system |

Fitting answers “how do we predict?”, tuning “how do we combine?”, calibration “how much should we trust it?”, and validation “how well does the completed procedure work on held-out companies?”. Reusing the same records for all these roles would make evaluation optimistic. Company IDs define groups: all records of one company stay together. The split is not stratified by industry, and record percentages can differ from group percentages.

A comprehension such as `[embeddings(d) for d in ...]` applies the same operation to each table. Tuple unpacking assigns the four returned values to four names. `LabelEncoder` assigns integer identifiers to industries, but class 2 is not “twice” class 1.

In [ ]:
# These ratios provide financial evidence: net profit margin, EBITDA margin and asset turnover.
financial_columns = ['net_profit_margin', 'ebitda_margin', 'asset_turnover']

# Training and inference share the same preparation; financials=None means finances are already joined or unavailable.
def prepare_companies(companies, financials=None):
    # When supplied, the separate finance table is authoritative and replaces any existing financial columns.
    if financials is not None:
        # Use ID as the index; reindex adds unavailable ratios as NaN, and infinities become missing values.
        financials = financials.set_index('id').reindex(columns=financial_columns).replace([np.inf, -np.inf], np.nan)
        # Group by the ID index: a median per ratio prevents duplicate join matches and reduces sensitivity to extreme records.
        financials = financials.groupby('id').median()
        # drop prevents ambiguous column suffixes; errors=ignore also accepts input without existing financial columns.
        companies = companies.drop(columns=financial_columns, errors='ignore').merge(
            # left preserves every company in its original order; many_to_one verifies at most one financial match per ID.
            financials, on='id', how='left', validate='many_to_one', sort=False)
    # union retains existing columns; reindex adds absent ratios as NaN; copy protects the input table from mutation.
    companies = companies.reindex(columns=companies.columns.union(financial_columns, sort=False)).copy()
    # Normalize infinities even for prejoined input; keep valid negative observations.
    companies[financial_columns] = companies[financial_columns].replace([np.inf, -np.inf], np.nan)
    # Return the prepared table without using industry labels to reconstruct financial information.
    return companies

# Load only the two trusted course pickle files: pickle must not be loaded from an untrusted source.
df = prepare_companies(pd.read_pickle('data/df_train.pkl'), pd.read_pickle('data/df_financials_train.pkl'))

# A small helper avoids repeating the same splitting procedure three times.
def split(companies, size):
    # next retrieves the single pair of position arrays; size is the held-out group fraction and seed 42 makes it repeatable.
    a, b = next(GroupShuffleSplit(n_splits=1, test_size=size, random_state=42).split(
        # ID defines the groups: two records from the same company cannot enter different subsets.
        companies, groups=companies['id']))
    # iloc selects positions in the table that was just split, rather than matching its index labels.
    return companies.iloc[a], companies.iloc[b]

# Reserve about 10% of company IDs for evaluating the completed system.
df_train, df_val = split(df, 0.1)
# Of the remaining 90%, use 90% for fitting: approximately 0.9 times 0.9 = 81% of the total.
df_fit, df_development = split(df_train, 0.1)
# Split the remaining 9% equally: about 4.5% for voting-weight selection and 4.5% for learning confidence.
df_tune, df_cal = split(df_development, 0.5)
# This comprehension counts records in the four subsets; record percentages are approximate when splitting groups.
display(pd.Series([len(d) for d in [df_fit, df_tune, df_cal, df_val]],
                  # The Series labels identify the purpose of each record count.
                  index=['Fitting', 'Tuning', 'Calibration', 'Validation'], name='Records'))

# Each company has a semantic vector; this helper builds the numerical matrix required by the models.
def embeddings(companies):
    # map converts each embedding value while retaining company order.
    values = companies['business_description_embedding'].map(
        # The lambda parses strings only; embeddings that are already lists or arrays remain usable.
        lambda value: ast.literal_eval(value) if isinstance(value, str) else value)
    # The matrix shape is (companies, embedding dimensions); float32 halves storage compared with float64.
    return np.array(values.tolist(), dtype=np.float32)

# Apply the same conversion to every subset to preserve coordinate meaning and numerical type.
X_fit, X_tune, X_cal, X_val = [embeddings(d) for d in [df_fit, df_tune, df_cal, df_val]]
# Learn class names from fitting companies; the integer codes identify industries but have no numerical ordering meaning.
le = LabelEncoder().fit(df_fit['industry'])
# Set union | and subset comparison <= verify that every held-out industry occurs in fitting.
assert set(df_tune['industry']) | set(df_cal['industry']) | set(df_val['industry']) <= set(le.classes_)
# transform reuses the fitted label mapping; each y contains one label for each row of its corresponding X.
y_fit, y_tune, y_cal, y_val = [le.transform(d['industry']) for d in [df_fit, df_tune, df_cal, df_val]]
# shape[1] counts embedding coordinates; len(le.classes_) counts industries, not companies.
print('Embedding dimensions:', X_fit.shape[1], '| Industries:', len(le.classes_))


## 2. Class imbalance and PCA

A common industry can dominate overall accuracy. Count companies by industry before interpreting the metrics. The bar chart makes imbalance visible; it does not correct it.

**Principal component analysis** finds linear combinations of coordinates that preserve the greatest variance. The first component retains the most variance; the second retains the greatest remaining variance while being orthogonal to the first. PCA does not use industry labels, so high variance does not necessarily imply useful class separation. Scikit-learn centers the coordinates before projection.

Retain 128 components from the earlier project to compress embeddings and reduce the cost of k-NN distance calculations. The scatterplot shows only two components; k-NN uses all 128 and logistic regression uses the original coordinates. We do not choose settings because a plot looks attractive or claim these settings are optimal for every dataset.

**Why fit PCA on fitting companies only?** Even without labels, learning directions from held-out companies would reveal information about their distribution. Apply only `transform` to the remaining subsets. `X_fit_pca[mask, 0]` selects companies in a particular industry and their first PCA coordinate. For probability matrices, `axis=1` means operating across the class columns within each company’s row.

In [ ]:
# Count industries within the training portion without using validation to choose model settings.
counts = df_train['industry'].value_counts()
# Sorted horizontal bars make rare classes and long industry names easier to compare.
counts.sort_values().plot.barh(figsize=(9, 6), title='Industry distribution in training data')
# The horizontal axis shows company counts, rather than confidence values or percentages.
plt.xlabel('Number of companies')
# The vertical axis identifies industries; compare bar lengths to inspect imbalance.
plt.ylabel('Industry')
# Adjust margins so labels fit inside the figure.
plt.tight_layout()
# Display the class-balance figure before constructing the PCA plot.
plt.show()

# Keep 128 directions of high variance, retaining the earlier project setting rather than selecting it on validation.
pca = PCA(n_components=128, random_state=42)
# fit_transform learns the mean and directions only from fitting embeddings, then projects those records.
X_fit_pca = pca.fit_transform(X_fit)
# transform applies the learned projection to held-out subsets; refitting here would introduce leakage.
X_tune_pca, X_cal_pca, X_val_pca = [pca.transform(X) for X in [X_tune, X_cal, X_val]]
# Use a separate figure with room for the points and industry legend.
plt.figure(figsize=(10, 6))
# One color per industry helps compare concentrations and overlap.
colors = sns.color_palette('husl', len(le.classes_))
# enumerate supplies each class code and its industry name in the same order used by y_fit.
for label, industry in enumerate(le.classes_):
    # The Boolean mask selects fitting companies from the current industry.
    mask = y_fit == label
    # Coordinates 0 and 1 are the first two components; small transparent points reveal dense regions.
    plt.scatter(X_fit_pca[mask, 0], X_fit_pca[mask, 1], s=6, alpha=0.3, color=colors[label], label=industry)
# PC1 combines original coordinates; it is not an individually identifiable financial ratio.
plt.xlabel('Principal component 1')
# PC2 is orthogonal to PC1 and retains the greatest remaining variance.
plt.ylabel('Principal component 2')
# The title identifies fitting data rather than suggesting independent evaluation.
plt.title('Fitting embeddings: first two PCA components')
# Move the legend outside the plot so it does not cover company clusters.
plt.legend(bbox_to_anchor=(1.02, 1), fontsize=7, markerscale=3)
# Adjust spacing for the external legend and long labels.
plt.tight_layout()
# Overlap in this two-dimensional view does not prove inseparability in 128 or 768 dimensions.
plt.show()
# Summing the first two variance ratios measures the variance displayed; [:2] selects positions 0 and 1.
print('Variance retained (2 / 128 components):', pca.explained_variance_ratio_[:2].sum(),
      # Summing all retained ratios measures variance preserved by 128 components, not accuracy or F1.
      pca.explained_variance_ratio_.sum())


## 3. Two classifiers and a simple combination

**Multiclass logistic regression.** Despite its name, this is a classifier here: it learns a linear score for each industry and converts scores to probabilities. Regularization discourages excessively large weights. `C` is inversely related to regularization strength; `C=10` does not mean ten classes. `max_iter=1000` limits optimization iterations, rather than guaranteeing 1,000 passes or defining 1,000 models.

**k-nearest neighbors.** For a new company, find 15 nearby fitting companies in the 128-dimensional PCA space and combine their industry labels. `weights='distance'` gives closer neighbors greater influence. With the default distance settings used here, distance is Euclidean. Fitting mainly retains data; searching neighbors during prediction can be the expensive step. Neighbor weights differ from the weight used to combine the two classifiers.

**Soft voting.** Compute `w × LR probabilities + (1-w) × k-NN probabilities`, then choose the industry with the largest value. This averages probability vectors rather than voting only on two predicted industry names. Compare three simple values of `w` on tuning F1 instead of performing a large search. Neither calibration nor validation selects this weight.

The settings are retained from the previous version, not newly optimized without the course datasets. Voting is not guaranteed to improve predictions: models can make correlated mistakes. Cache their probability matrices to make comparisons efficient.

In [ ]:
# Logistic regression learns from all coordinates; C=10 inversely controls regularization and 1000 limits solver iterations.
lr = LogisticRegression(C=10, max_iter=1000).fit(X_fit, y_fit)
# k-NN uses the 128 PCA coordinates, 15 neighbors, distance-weighted votes and two parallel jobs.
knn = KNeighborsClassifier(n_neighbors=15, weights='distance', n_jobs=2).fit(X_fit_pca, y_fit)
# Check probability-column alignment before voting: column j must refer to the same industry in both models.
assert np.array_equal(lr.classes_, knn.classes_) and np.array_equal(lr.classes_, np.arange(len(le.classes_)))
# Each predict_proba result has shape (companies, industries); retain results to avoid repeated computation.
proba_tune_lr, proba_cal_lr, proba_val_lr = [lr.predict_proba(X) for X in [X_tune, X_cal, X_val]]
# k-NN needs PCA coordinates rather than original embeddings; the three held-out subsets remain separate.
proba_tune_knn, proba_cal_knn, proba_val_knn = [knn.predict_proba(X) for X in [X_tune_pca, X_cal_pca, X_val_pca]]
# Compare three simple logistic-regression weights; the corresponding k-NN weight is 1-w.
weights = [0.25, 0.5, 0.75]
# For each weight, average the probability vectors; argmax(axis=1) selects the largest class within each row.
tuning_f1 = [f1_score(y_tune, (w * proba_tune_lr + (1 - w) * proba_tune_knn).argmax(axis=1),
                      # Weighted F1 averages industry F1 values by support; only tuning companies select this weight.
                      average='weighted') for w in weights]
# np.argmax locates the best tuning score; indexing weights retrieves its weight, not the score itself.
lr_weight = weights[np.argmax(tuning_f1)]
# The table makes the comparison between the three weights visible.
display(pd.DataFrame({'LR weight': weights, 'Tuning weighted F1': tuning_f1}))
# Report the weight actually selected instead of assuming it will always be 0.25.
print('Selected LR weight:', lr_weight)


## 4. Financial evidence and correctness calibration

Financial information **does not change the industry selected by voting**. It helps estimate how plausible that industry is. For every industry and ratio, retain sorted fitting observations. We do not automatically assume normality: negative margins, long tails and outliers make blanket Gaussian or log-normal assumptions questionable.

A percentile near .5 is central; one near 0 or 1 is in a tail. `2 × min(p,1-p)` gives compatibility 1 at the center and low values in the tails. For example, p=.05 gives .10. This is an **empirical compatibility index**, not “10% probability that the industry is correct” and not a formally combined p-value. An unusual company can have a correct industry, and a financially typical company can be misclassified.

`searchsorted` finds positions in a sorted reference without scanning every observation again. For reference `[1,2,2,4]` and observed value 2, the left position is 1 and the right position is 3. Their midpoint treats ties symmetrically. The small offset in the formula keeps the percentile inside (0,1), including observations beyond the reference minimum or maximum. Profiles with fewer than 20 observations are omitted. Average scores across available ratios: three if three exist, two if two exist. With none, use a .5 placeholder together with coverage 0.

### A second classification problem

The main classifiers predict **industry**. The calibrator predicts **correct industry prediction=1 or incorrect prediction=0**. Learning this requires companies the main classifiers did not see during fitting. The final calibrator receives five features:

1. Maximum vote, measuring the strength of the leading class.
2. Gap between the first and second votes, measuring ambiguity.
3. Agreement between the two classifiers, as additional evidence.
4. Mean financial compatibility.
5. Fraction of usable ratios, identifying missing financial evidence.

The text-only calibrator uses the first three features, providing a direct comparison for the extra financial evidence. Both are simple binary logistic regressions. They minimize log-loss, not Brier directly; evaluation must check whether Brier also improves. The `[:,1]` probability column represents class correct=1. Keep the base classifiers frozen afterward: refitting them would change the system on which confidence was learned.

`predict_industry(companies, financials=None)` prepares inputs and returns `(industry_names, confidence)`. A separate finance table replaces prejoined financial columns; otherwise unavailable ratios become `NaN`. The true `industry` is not required. Inference reuses fitted objects rather than learning from the new companies.

In [ ]:
# For each industry-ratio pair, retain sorted nonmissing financial observations from fitting companies only.
profiles = {(industry, column): np.sort(df_fit.loc[df_fit['industry'] == industry, column].dropna())
            # This dictionary comprehension covers all pairs; each key is a tuple containing industry and ratio.
            for industry in le.classes_ for column in financial_columns}

# Build confidence features from predictions and finances without reading the true industries of these companies.
def confidence_features(companies, proba_lr, proba_knn):
    # Soft voting averages aligned class probabilities; each row still sums to one.
    votes = lr_weight * proba_lr + (1 - lr_weight) * proba_knn
    # Select the industry with the largest vote; financial evidence will not change this class prediction.
    prediction = votes.argmax(axis=1)
    # Sort probabilities within each row to obtain the largest and second-largest values.
    top = np.sort(votes, axis=1)
    # Decode class codes to choose the financial profile of the PREDICTED industry.
    predicted_industries = le.inverse_transform(prediction)
    # One accumulator sums compatibility and the other counts usable ratios for each company.
    total, available = np.zeros(len(companies)), np.zeros(len(companies))
    # Visit every industry, updating only companies assigned to that industry by the vote.
    for industry in le.classes_:
        # Treat the three financial ratios separately because their scales and interpretations differ.
        for column in financial_columns:
            # Reference distributions always come from fitting, never from calibration or validation companies.
            reference = profiles[industry, column]
            # Skip profiles with fewer than 20 observations: this practical minimum is retained from the baseline.
            if len(reference) < 20:
                # continue skips this profile without removing companies or stopping the remaining ratios.
                continue
            # Extract ratios in company-row order so they remain aligned with model probabilities.
            values = companies[column].to_numpy()
            # & combines two requirements: the predicted industry matches this profile, and the observed value is finite.
            mask = (predicted_industries == industry) & np.isfinite(values)
            # The left insertion position counts reference values strictly below the observed ratio.
            left = np.searchsorted(reference, values[mask], side='left')
            # The right insertion position includes equal values; averaging the positions handles ties symmetrically.
            right = np.searchsorted(reference, values[mask], side='right')
            # The midpoint rank and small offset keep percentiles inside (0,1), including values beyond the reference extremes.
            percentile = (0.5 * (left + right) + 0.5) / (len(reference) + 1)
            # Compatibility is 2 times min(p,1-p): 1 at p=.5 and .1 at p=.05; it is evidence, not correctness probability.
            total[mask] += 2 * np.minimum(percentile, 1 - percentile)
            # Count only ratios actually used; missing evidence must not be mistaken for incompatibility.
            available[mask] += 1
    # Average usable scores; keep .5 when none exist. where prevents division by zero without inventing observations.
    compatibility = np.divide(total, available, out=np.full(len(companies), 0.5), where=available > 0)
    # column_stack creates five columns: maximum vote and gap to the runner-up describe strength and ambiguity.
    features = np.column_stack([top[:, -1], top[:, -1] - top[:, -2],
                               # The third column records model agreement; agreement does not guarantee correctness.
                               proba_lr.argmax(axis=1) == proba_knn.argmax(axis=1),
                               # The last columns are mean compatibility and coverage, distinguishing observed .5 from a missing-data placeholder.
                               compatibility, available / len(financial_columns)])
    # Return predicted class codes with shape (n,) and confidence features with shape (n,5), preserving row order.
    return prediction, features

# Build features on calibration companies unseen during the base classifiers’ fitting.
pred_cal, features_cal = confidence_features(df_cal, proba_cal_lr, proba_cal_knn)
# The new target is correct=1 or incorrect=0, not industry. True labels teach confidence only at this training stage.
correct_cal = (pred_cal == y_cal).astype(int)
# The text-only comparison learns from the first three columns: probabilities, margin and agreement.
text_calibrator = LogisticRegression().fit(features_cal[:, :3], correct_cal)
# The final calibrator also sees financial evidence and coverage; the industry classifiers remain frozen.
finance_calibrator = LogisticRegression().fit(features_cal, correct_cal)

# This interface predicts new companies without requiring their true industry and returns two vectors.
def predict_industry(companies, financials=None):
    # Reuse training-time preparation; separate finances take precedence and absent ratios become NaN.
    companies = prepare_companies(companies, financials)
    # Convert embeddings without changing coordinate meaning or company order.
    X = embeddings(companies)
    # Logistic regression receives original X, while k-NN receives pca.transform(X); all models are already fitted.
    prediction, features = confidence_features(companies, lr.predict_proba(X), knn.predict_proba(pca.transform(X)))
    # Return industry names and the calibrator probability of class 1, meaning estimated prediction correctness.
    return le.inverse_transform(prediction), finance_calibrator.predict_proba(features)[:, 1]


## 5. Evaluating the required classifiers

Models and voting weight are already fixed. Validation measures the completed procedure without selecting settings after inspecting its results. Its metrics describe this split, not guaranteed performance on the private test set.

For one industry, **precision = TP/(TP+FP)** measures how many predictions of that industry are correct. **Recall = TP/(TP+FN)** measures how many actual companies in that industry are recognized. **F1 = 2 × precision × recall / (precision+recall)** balances the two. `support` is the number of actual class members in the evaluation. Weighted F1 weights class F1 values by support; macro F1 takes their simple average. Accuracy is the fraction of all records classified correctly.

The confusion matrix uses true industries as rows and predicted industries as columns. Row normalization gives comparable fractions for small and large industries; these values are not precision. The five most frequent confusion pairs instead use raw counts and can be dominated by common industries.

Correlation between industry size and `1-recall` describes an association, not proof that rarity alone causes difficulty. Shared errors show where both models struggle; model-specific errors indicate possible complementary information. Always state which classifier performs better **under which metric and on which sample**.

In [ ]:
# Compare the two required classifiers on the same validation companies; argmax selects their predicted class codes.
predictions = {'Logistic regression': proba_val_lr.argmax(axis=1), 'k-NN': proba_val_knn.argmax(axis=1)}
# Collect one metric row per model without changing the already fitted classifiers.
results = []
# items supplies each model name and prediction vector for a symmetric comparison.
for name, prediction in predictions.items():
    # Accuracy measures the correct fraction; weighted F1 averages industry F1 values using validation support.
    results.append([name, accuracy_score(y_val, prediction), f1_score(y_val, prediction, average='weighted'),
                    # Macro F1 weights every industry equally, revealing weaknesses in rare classes that contribute little to weighted F1.
                    f1_score(y_val, prediction, average='macro')])
    # The report calculates precision, recall, F1 and support for every industry in the fixed label mapping.
    report = pd.DataFrame(classification_report(y_val, prediction, labels=np.arange(len(le.classes_)),
                         # zero_division=0 handles undefined ratios; .T puts classes on rows and .loc keeps the industry rows only.
                         target_names=le.classes_, output_dict=True, zero_division=0)).T.loc[le.classes_]
    # Print the model name to distinguish the following report.
    print(name)
    # Sort by increasing F1 so the weakest industries appear first.
    display(report.sort_values('f1-score'))
    # reindex aligns industries; 1-recall is the error fraction among true members of a class. Correlation is not causation.
    print('Industry size / error-rate correlation:', counts.reindex(report.index).corr(1 - report['recall']))
    # Raw confusion counts use true industries as rows and predicted industries as columns; diagonal entries are correct.
    cm = confusion_matrix(y_val, prediction, labels=np.arange(len(le.classes_)))
    # Attach industry names to make the class pairs readable without looking up integer codes.
    pairs = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
    # Zero this count table’s diagonal to rank mistakes rather than successful predictions.
    np.fill_diagonal(pairs.values, 0)
    # stack lists true-predicted pairs; sort them to show the five most frequent confusions.
    display(pairs.stack().sort_values(ascending=False).head(5).rename('Misclassifications'))
    # Use a separate figure per model with enough space for long industry names.
    plt.figure(figsize=(10, 8))
    # Recompute a confusion matrix normalized by true class: each row shows fractions rather than raw counts.
    sns.heatmap(confusion_matrix(y_val, prediction, labels=np.arange(len(le.classes_)), normalize='true'),
                # Use the same 0–1 color scale so both models and rare/common industries remain comparable.
                xticklabels=le.classes_, yticklabels=le.classes_, cmap='Blues', vmin=0, vmax=1)
    # Columns show the industry assigned by the classifier.
    plt.xlabel('Predicted industry')
    # Rows show the true industry; off-diagonal values identify where that class loses recall.
    plt.ylabel('True industry')
    # The title identifies the classifier and the meaning of the normalization.
    plt.title(name + ': fraction of each true industry')
    # Adjust spacing before displaying the heatmap.
    plt.tight_layout()
    # Display this model’s confusion plot before processing the next model.
    plt.show()
# The summary table compares overall metrics with one labeled row per classifier.
comparison = pd.DataFrame(results, columns=['Model', 'Accuracy', 'Weighted F1', 'Macro F1']).set_index('Model')
# Read weighted and macro F1 separately: they assign different importance to small industries.
display(comparison)
# idxmax identifies the stronger of these two models on this validation and metric, not a universal winner.
print('Higher validation weighted F1:', comparison['Weighted F1'].idxmax())
# Boolean vectors identify the individual companies misclassified by each model.
errors_lr, errors_knn = predictions['Logistic regression'] != y_val, predictions['k-NN'] != y_val
# Measure shared and model-specific errors; complementary errors can make voting useful.
print('Both wrong / only LR wrong / only k-NN wrong:',
      # & intersects masks, ~ negates them and sum counts True values: both wrong, only LR wrong, only k-NN wrong.
      (errors_lr & errors_knn).sum(), (errors_lr & ~errors_knn).sum(), (~errors_lr & errors_knn).sum())


## 6. Official score and financial distributions

The current skeleton defines:

**score = weighted F1 × (1 − Brier)**

**Brier = mean((confidence − correctness)²)**, where correctness is 0 or 1 for the selected industry.

A correct prediction with confidence .9 contributes .01 to Brier; an incorrect prediction with confidence .9 contributes .81. Always reporting 1 heavily penalizes mistakes; always reporting low confidence penalizes correct predictions. Confidence does not enter the F1 calculation. This Brier on prediction correctness differs from the usual multiclass Brier using the entire probability vector.

All three ensemble variants predict the same industries, so their F1 values are identical. Calibration can change Brier and the official score only. Show comparisons without selecting the final candidate from validation: the financial candidate was fixed in advance. Its extra benefit might be small or absent and must be reported honestly.

Because Brier is nonnegative, score cannot exceed F1. No confidence transformation can compensate for industry F1 below the target. The old arbitrary `confidence ** 6` transformation was associated with a different scorer and has no general justification here.

Boxplots show fitting distributions: medians, quartiles and dispersion. Hiding extreme dots does not remove those observations from the data or reference profiles. `symlog` displays valid negative ratios alongside long tails. Overlapping financial distributions explain why financial evidence is supporting information rather than proof of an industry.

In [ ]:
# Follow the current skeleton’s scorer, rather than the older F1 calculation using confidence as sample weights.
def confidence_score(y_true, y_pred, confidence):
    # Convert possible input lists to comparable arrays and represent confidence numerically as floating-point values.
    y_true, y_pred, confidence = np.asarray(y_true), np.asarray(y_pred), np.asarray(confidence, dtype=float)
    # Require one prediction and confidence per true label; no difficult records are excluded.
    assert len(y_true) == len(y_pred) == len(confidence)
    # Confidence must be within [0,1]; NaN also fails this check.
    assert np.all((confidence >= 0) & (confidence <= 1))
    # Brier is mean squared error against correctness; score = weighted F1 times (1-Brier), so score cannot exceed F1.
    return f1_score(y_true, y_pred, average='weighted') * (1 - np.mean((confidence - (y_pred == y_true)) ** 2))

# Build validation features using the fixed vote without refitting anything.
pred_val, features_val = confidence_features(df_val, proba_val_lr, proba_val_knn)
# Raw ensemble confidence is the largest vote, stored in the first feature column.
confidence_methods = {'Raw ensemble': features_val[:, 0],
                      # Text-calibrated confidence is the estimated probability of correctness from the first three features.
                      'Text-calibrated ensemble': text_calibrator.predict_proba(features_val[:, :3])[:, 1],
                      # Financial-calibrated confidence uses the same industry predictions and all five features.
                      'Financial-calibrated ensemble': finance_calibrator.predict_proba(features_val)[:, 1]}
# Each entry stores (predictions, confidence); the three ensembles therefore have identical industry F1.
methods = {name: (pred_val, c) for name, c in confidence_methods.items()}
# For standalone logistic regression, use its largest class probability as raw confidence.
methods['Logistic regression'] = (predictions['Logistic regression'], proba_val_lr.max(axis=1))
# Apply the same raw-confidence definition to k-NN for a readable comparison.
methods['k-NN'] = (predictions['k-NN'], proba_val_knn.max(axis=1))
# Collect metrics for every method without selecting the final candidate by validation score.
bonus_results = []
# Tuple unpacking separates the method name, predicted industries and confidence vector.
for name, (prediction, confidence) in methods.items():
    # F1 depends on predicted labels rather than how confident the model claims to be.
    bonus_results.append([name, f1_score(y_val, prediction, average='weighted'),
                          # Boolean correctness behaves as 0/1 here; confident mistakes receive a large squared penalty.
                          np.mean((confidence - (prediction == y_val)) ** 2),
                          # Evaluate the same official scorer for every method; financial evidence must demonstrate its benefit empirically.
                          confidence_score(y_val, prediction, confidence)])
# Showing F1 and Brier separately explains which component changes the final score.
display(pd.DataFrame(bonus_results, columns=['Method', 'Weighted F1', 'Brier', 'Official score']).set_index('Method'))
# Nonmissing counts and medians summarize financial coverage and typical values within fitting industries only.
display(df_fit.groupby('industry')[financial_columns].agg(['count', 'median']))
# Use one plot per ratio because the economic interpretation and numerical scale differ.
for column in financial_columns:
    # Choose figure dimensions that leave room for many industry names.
    plt.figure(figsize=(9, 6))
    # The boxplot displays medians and interquartile ranges; showfliers=False hides extreme dots in the plot only.
    sns.boxplot(data=df_fit, x=column, y='industry', showfliers=False)
    # symlog is approximately linear near zero and logarithmic in the tails, allowing valid negative margins to appear.
    plt.xscale('symlog', linthresh=0.1)
    # Make the ratio name readable and state the axis scaling.
    plt.xlabel(column.replace('_', ' ') + ' (symmetric log scale)')
    # Industries define the financial distributions being compared.
    plt.ylabel('Industry')
    # Identify the current ratio and clarify that these are fitting distributions.
    plt.title('Fitting financial distributions: ' + column.replace('_', ' '))
    # Adjust the figure layout without deleting observations or modifying the model.
    plt.tight_layout()
    # Look for overlap and dispersion: finances provide evidence rather than a deterministic rule for industry.
    plt.show()


## 7. Reliability, human review and the prediction interface

A calibrated model is correct on roughly 80% of many predictions assigned confidence .8. Calibration cannot be verified from one company. Group confidence into ten bins and compare mean confidence with accuracy, including counts: small bins provide unstable estimates. Calibration is distinct from classification quality; a weak classifier can still estimate its uncertainty honestly.

The .8 threshold is an illustrative human-review policy, not an official bonus threshold. More review requires more work. Flagged companies **remain in every score**; difficult cases are not removed to improve evaluation. Report workload and captured errors without assuming a reviewer automatically corrects every mistake.

The inference example removes `industry` before calling the model. Embeddings and, when available, financial ratios are sufficient. Outputs follow the original row order. Future companies must not be used to refit PCA or the classifiers during inference.

Finally, inspect repeated descriptions. Distinct IDs can have exactly identical embeddings. Grouping by ID prevents one company entering two subsets but does not guarantee wholly new descriptions. Separate accuracy for seen and unseen embeddings makes that limitation visible. Exact matching of stored embeddings is a diagnostic, not a search for every possible semantic similarity.

In [ ]:
# Use the financial candidate fixed in advance, rather than selecting the best-looking calibrator on validation.
confidence_val = confidence_methods['Financial-calibrated ensemble']
# Pair each reported confidence with whether the industry prediction proved correct.
outcomes = pd.DataFrame({'Confidence': confidence_val, 'Correct': pred_val == y_val})
# Eleven boundaries from 0 to 1 define ten bins; include_lowest includes confidence exactly zero.
outcomes['Bin'] = pd.cut(confidence_val, np.linspace(0, 1, 11), include_lowest=True)
# Group by confidence bin; observed=True omits empty groups without inventing their accuracy.
reliability = outcomes.groupby('Bin', observed=True).agg(
    # Count shows sample size; the mean of Boolean Correct is the observed correct fraction, or accuracy.
    Count=('Correct', 'size'), Mean_confidence=('Confidence', 'mean'), Accuracy=('Correct', 'mean'))
# A bin with very few cases provides weak evidence about calibration.
display(reliability)
# The diagonal is the ideal relation: mean confidence equals observed accuracy.
plt.plot([0, 1], [0, 1], '--', color='gray', label='Perfect calibration')
# Points below the diagonal indicate overconfidence; points above it indicate underconfidence in this sample.
plt.plot(reliability['Mean_confidence'], reliability['Accuracy'], 'o-', label='Validation bins')
# The x-axis shows what the model claimed before the true outcome was known.
plt.xlabel('Mean predicted confidence')
# The y-axis shows the fraction that was actually correct.
plt.ylabel('Observed accuracy')
# The title concerns confidence quality, rather than a decision boundary between industries.
plt.title('Confidence and accuracy')
# Distinguish the ideal reference from observed validation bins.
plt.legend()
# Adjust the plot before displaying it.
plt.tight_layout()
# Use the plot together with bin counts; it does not establish perfect calibration by itself.
plt.show()
# Below .8, suggest human review: an illustrative policy, not a professor-mandated or validation-optimized threshold.
review = confidence_val < 0.8
# Add a review flag without removing any predictions from the score.
outcomes['Human review'] = review
# Compare workload and accuracy between flagged and unflagged companies.
display(outcomes.groupby('Human review').agg(Count=('Correct', 'size'), Accuracy=('Correct', 'mean')))
# The flag mean measures review workload; intersecting the flag with errors counts captured mistakes.
print('Flagged share:', review.mean(), '| Errors captured:', np.sum(review & (pred_val != y_val)),
      # Report total errors to show how many remain unflagged, without assuming human review automatically fixes them.
      'of', np.sum(pred_val != y_val))
# Remove the true industry and select three examples to demonstrate the inference interface.
example = df_val.drop(columns='industry').head(3)
# Return industry names and confidence values without requiring the true target.
industries, confidences = predict_industry(example)
# Attach IDs to the outputs to make preservation of input order explicit.
display(pd.DataFrame({'ID': example['id'].to_numpy(), 'Predicted industry': industries, 'Confidence': confidences}))
# isin detects exact embeddings already present in fitting; different company IDs can share the same description.
seen_embedding = df_val['business_description_embedding'].isin(df_fit['business_description_embedding'])
# Compare repeated-description status with the final candidate’s observed correctness.
display(pd.DataFrame({'Embedding seen during fitting': seen_embedding.to_numpy(), 'Correct': pred_val == y_val})
        # Separate accuracy for seen/unseen embeddings exposes a limitation of splitting by company ID alone.
        .groupby('Embedding seen during fitting').agg(Count=('Correct', 'size'), Accuracy=('Correct', 'mean')))
# This duration covers the current run up to this point; cloud timing does not certify another machine’s runtime.
print('Whole notebook elapsed seconds:', round(perf_counter() - started, 1))


## Explaining the project in the oral exam

A possible opening answer is: “I use supplied embeddings to represent the meaning of business descriptions. I separate fitting, voting-weight selection, confidence calibration and evaluation. I compare logistic regression on full embeddings with k-NN on 128 PCA components. I average their probabilities with a weight chosen on tuning data. Another logistic regression learns the probability that the industry prediction is correct from textual and financial evidence. I evaluate both industry labels and confidence without using the private test set.”

Explain the reasons behind the functions rather than memorizing their names:

- **Why PCA rather than two arbitrary coordinates?** It orders directions by retained variance. Two components serve visualization; 128 serve k-NN distance calculations.
- **Why logistic regression and k-NN?** One learns a global linear relationship and the other uses local neighborhoods. Both are required and may offer complementary predictions.
- **Why not guarantee maximum F1?** The private dataset is unknown. Available information, chosen settings and evaluation design limit what the results establish.
- **Why four subsets?** Learning and selection stages need separate roles while evaluation remains independent of those choices.
- **Why financial ranks?** They yield bounded comparable evidence without imposing normality on financial distributions containing negative values and long tails.
- **Why avoid calling compatibility a p-value?** Averaging empirical tail-compatibility indices does not provide a formal statistical test with a guaranteed p-value interpretation.
- **Why another logistic regression?** Its target is correctness rather than industry; it converts confidence-related evidence into a learned probability.
- **Why show macro F1?** Small industries can contribute little to weighted F1 but receive equal importance under macro F1.
- **Why does finance not improve F1 here?** It modifies confidence only, leaving the ensemble’s predicted labels unchanged.
- **What would you improve with more data and time?** Check stability across additional splits and new descriptions, then evaluate a small number of measurable changes. Do not claim those experiments have already been run.

Adapt plot interpretations to an actual execution. Historical results are retained in `archive/`; without the course datasets, these cells do not establish newly measured real-data performance. Cloud timing does not certify laptop runtime, and no private score or bonus is guaranteed. AI assistance is disclosed. Use `EXPLANATION.md` for the complete study guide.